In [33]:
import pandas as pd
import numpy as np
import json
import os

In [34]:
bgg_data = pd.DataFrame()
# Load all JSON files from data/groups folder
data_files = []
for filename in os.listdir('data/groups'):
    if filename.endswith('.json'):
        with open(os.path.join('data/groups', filename), 'r', encoding='utf-8') as f:
            data_files.append(json.load(f))

bgg_data = pd.DataFrame([item for sublist in data_files for item in (sublist if isinstance(sublist, list) else [sublist])])

In [35]:
bgg_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 177318 entries, 0 to 177317
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   row_id                177318 non-null  str   
 1   type                  177318 non-null  str   
 2   boardgame             177318 non-null  str   
 3   description           177280 non-null  str   
 4   min_players           177318 non-null  str   
 5   max_players           177318 non-null  str   
 6   suggested_numplayers  177318 non-null  object
 7   min_playtime          177318 non-null  str   
 8   max_playtime          177318 non-null  str   
 9   playingtime           177318 non-null  str   
 10  minimum_age           177318 non-null  str   
 11  release_year          177318 non-null  str   
 12  average_rating        177318 non-null  str   
 13  num_of_ratings        177318 non-null  str   
 14  weight                177318 non-null  str   
 15  num_of_weights        177318

In [36]:
numericos = ['release_year', 'min_players', 'max_players', 'playingtime', 'minimum_age', 'num_of_ratings','weight', 
             'num_of_weights', 'average_rating', 'bayes_average', 'std_deviation','owned', 'trading', 'wanting', 'wishing']
for col in numericos:
    bgg_data[col] = pd.to_numeric(bgg_data[col], errors='coerce')
bgg_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 177318 entries, 0 to 177317
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   row_id                177318 non-null  str    
 1   type                  177318 non-null  str    
 2   boardgame             177318 non-null  str    
 3   description           177280 non-null  str    
 4   min_players           177318 non-null  int64  
 5   max_players           177318 non-null  int64  
 6   suggested_numplayers  177318 non-null  object 
 7   min_playtime          177318 non-null  str    
 8   max_playtime          177318 non-null  str    
 9   playingtime           177318 non-null  int64  
 10  minimum_age           177318 non-null  int64  
 11  release_year          177318 non-null  int64  
 12  average_rating        177318 non-null  float64
 13  num_of_ratings        177318 non-null  int64  
 14  weight                177318 non-null  float64
 15  num_of_weig

In [37]:
# Guardamos el número de filas antes de aplicar los filtros para poder comparar después
filas_prefiltro = bgg_data.shape[0]

# Filtramos juegos con un número máximo de jugadores razonable (<300)
bgg_data = bgg_data[(bgg_data["max_players"]<=100)]

# Filtramos juegos con duración (playingtime) plausibles (<30000 minutos)
bgg_data = bgg_data[(bgg_data["playingtime"]<30000) & (bgg_data["playingtime"]>0)]

# Filtramos juegos con edad mínima razonable (<22 años)
bgg_data = bgg_data[(bgg_data["minimum_age"]<22)]

# Excluimos registros con años de lanzamiento futuros (<= 2026)
bgg_data = bgg_data[(bgg_data["release_year"]<=2026)]

# Exigimos un mínimo de valoraciones para mayor robustez (>30)
bgg_data = bgg_data[(bgg_data["num_of_ratings"]>30)]

# Excluimos juegos sin valoración promedio  y peso validos (>0)
bgg_data = bgg_data[(bgg_data["average_rating"]>0) & (bgg_data["weight"]>0)]

# Contamos las filas después de aplicar los filtros
filas_postfiltro = bgg_data.shape[0]

# Mostramos resumen del efecto de los filtros: antes, después, eliminadas y porcentaje eliminado
print(f"Filas antes del filtro: {filas_prefiltro}")
print(f"Filas después del filtro: {filas_postfiltro}")
print(f"Filas eliminadas: {filas_prefiltro - filas_postfiltro}")
print(f"Porcentaje de filas eliminadas: {((filas_prefiltro - filas_postfiltro) / filas_prefiltro) * 100:.2f}%")

Filas antes del filtro: 177318
Filas después del filtro: 36959
Filas eliminadas: 140359
Porcentaje de filas eliminadas: 79.16%


In [38]:
print(f"Juego más nuevo de la base de datos: {bgg_data.loc[bgg_data['release_year'].idxmax()]['boardgame']} = {bgg_data['release_year'].max()}")
print(f"Juego más antiguo de la base de datos: {bgg_data.loc[bgg_data['release_year'].idxmin()]['boardgame']} = {bgg_data['release_year'].min()}")

Juego más nuevo de la base de datos: Nippon: Zaibatsu = 2026
Juego más antiguo de la base de datos: Senet = -3500


Columnas que son object y que son candidatas a ser externalizadas en tablas aparte:

In [39]:
bgg_data[['suggested_numplayers','ranks', "categories", "mechanisms", "family", "designers", "artists","publishers"]] 

,suggested_numplayers,ranks,categories,mechanisms,family,designers,artists,publishers
0,"{'Best': '4', 'Recommended': '2', 'Not Recomme...","{'boardgame': '1', 'strategygames': '1'}","[Age of Reason, Economic, Industry / Manufactu...","[Chaining, End Game Bonuses, Hand Management, ...","[Cities: Birmingham (England), Components: Map...","[Gavan Brown, Matt Tolman, Martin Wallace]","[Gavan Brown, Lina Cossette, David Forest, Gui...","[Roxley, Arclight Games, Board Game Rookie, Bo..."
1,"{'Best': '2', 'Recommended': '3', 'Not Recomme...","{'boardgame': '2', 'strategygames': '2'}","[Animals, Card Game, Environmental]","[Contracts, End Game Bonuses, Events, Grid Cov...","[Animals: Okapi, Components: Hexagonal Tiles, ...",[Mathias Wigge],"[Steffen Bieker, Loïc Billiau, Dennis Lohausen]","[Feuerland Spiele, Capstone Games, CMON Global..."
2,"{'Best': '4', 'Recommended': '2', 'Not Recomme...","{'boardgame': '3', 'thematic': '1', 'strategyg...","[Environmental, Medical]","[Action Points, Cooperative Game, Hand Managem...","[Components: Map (Global Scale), Components: M...","[Rob Daviau, Matt Leacock]",[Chris Quilliams],"[Z-Man Games, Asterion Press, Devir, Filosofia..."
3,"{'Best': '3', 'Recommended': '2', 'Not Recomme...","{'boardgame': '4', 'thematic': '2', 'strategyg...","[Adventure, Exploration, Fantasy, Fighting, Mi...","[Action Queue, Action Retrieval, Campaign / Ba...","[Category: Dungeon Crawler, Components: Map (C...",[Isaac Childres],"[Alexandr Elichev, Lucile Mathieu, Josh T. McD...","[Cephalofair Games, Albi, Albi Polska, Arcligh..."
4,"{'Best': '4', 'Recommended': '3', 'Not Recomme...","{'boardgame': '5', 'strategygames': '4'}","[Movies / TV / Radio theme, Novel-based, Scien...","[Automatic Resource Growth, Card Play Conflict...","[Books: Dune, Game: Dune: Imperium, Misc: Long...",[Paul Dennen],"[Clay Brooks, Derek Herring, Raul Ramos, Nate ...","[Dire Wolf, Arclight Games, Broadway Toys LTD,..."
...,...,...,...,...,...,...,...,...
177313,"{'Best': '2', 'Recommended': '3', 'Not Recomme...","{'boardgame': '497', 'strategygames': '255'}","[Animals, Puzzle]","[Deck, Bag, and Pool Building, End Game Bonuse...","[Components: 3-Dimensional (3D), Components: M...",[Jamie Bloom],[Sophia Kang],"[Bad Comet, Delirium Games, DSV Games, GémKlub..."
177314,"{'Best': '3', 'Recommended': '2', 'Not Recomme...","{'boardgame': '499', 'thematic': '101'}","[Fighting, Horror, Miniatures, Science Fiction...","[Action Points, Cooperative Game, Dice Rolling...","[Category: Dungeon Crawler, Components: Miniat...","[Raphaël Guiton, Jean-Baptiste Lullien, Nicola...","[Édouard Guiton, Thierry Masson, Eric Nouhaut]","[CMON Global Limited, Guillotine Games, ADC Bl..."
177315,"{'Best': '2', 'Recommended': '2+', 'Not Recomm...","{'boardgame': '500', 'cgs': '20'}","[Card Game, Fantasy, Fighting]","[Deck Construction, Dice Rolling, Grid Movemen...","[Category: Two-Player Fighting Games, Componen...","[Bryan Pope, Benjamin Pope]","[Drew Baker, Tiziano Baracchi, Claire Beard, L...","[Arcane Wonders, Asterion Press, Devir, Game H..."
177316,"{'Best': '6', 'Recommended': '4', 'Not Recomme...","{'boardgame': '489', 'partygames': '10', 'fami...","[Animals, Dice, Racing]","[Open Drafting, Race, Roll / Spin and Move, Tr...","[Animals: Bears, Animals: Elephants, Animals: ...","[Richard Garfield, Takashi Ishida]",[Angela Kirkwood],"[CMYK, Geekach LLC, GémKlub Kft., IELLO]"


Generamos tablas accesorias para eliminar las listas dentro de las columnas

In [40]:
game_designers = bgg_data[['row_id','designers']]
game_designers = game_designers.explode('designers')
bgg_data.drop(columns=['designers'], inplace=True)

game_artists = bgg_data[['row_id','artists']]
game_artists = game_artists.explode('artists')
bgg_data.drop(columns=['artists'], inplace=True)

game_mechanics = bgg_data[['row_id','mechanisms']]
game_mechanics = game_mechanics.explode('mechanisms')
bgg_data.drop(columns=['mechanisms'], inplace=True)


game_categories = bgg_data[['row_id','categories']]
game_categories = game_categories.explode('categories')
bgg_data.drop(columns=['categories'], inplace=True)

game_family = bgg_data[['row_id','family']]
game_family = game_family.explode('family')
bgg_data.drop(columns=['family'], inplace=True)

game_publishers = bgg_data[['row_id','publishers']]
game_publishers = game_publishers.explode('publishers')
bgg_data.drop(columns=['publishers'], inplace=True)
bgg_data.shape

(36959, 24)

En el caso de las columnas con diccionarios, el proceso es diferente pero obtenemos también las tablas accesorias

In [41]:
# Vectorized expansion usando json_normalize (mucho más rápido que apply(pd.Series))
suggested_players_expanded = pd.json_normalize(bgg_data['suggested_numplayers'])
suggested_numplayers_table = pd.concat([bgg_data[['row_id']].reset_index(drop=True), suggested_players_expanded.reset_index(drop=True)], axis=1)
bgg_data.drop(columns=['suggested_numplayers'], inplace=True)
print(suggested_numplayers_table.head(10))
# Ranks: normalizar y convertir a numérico por columnas de forma robusta
ranks_expanded = pd.json_normalize(bgg_data['ranks'])

for col in ranks_expanded.columns:
    # intentar convertir a numérico; si son enteros convertimos a Int64 nullable
    ranks_expanded[col] = pd.to_numeric(ranks_expanded[col], errors='coerce')
    try:
        ranks_expanded[col] = ranks_expanded[col].astype('Int64')
    except Exception:
        # si no puede convertirse a Int64 (p.ej. contiene textos como 'Not Ranked'), dejar en numérico (float/NaN)
        pass

ranks_table = pd.concat([bgg_data[['row_id']].reset_index(drop=True), ranks_expanded.reset_index(drop=True)], axis=1)
bgg_data.drop(columns=['ranks'], inplace=True)

ranks_table.head(10)

   row_id Best Recommended Not Recommended
0  224517    4           2               1
1  342942    2           3              4+
2  161936    4           2              4+
3  174430    3           2              4+
4  397598    4           3               5
5  316554    3           2              4+
6  233078    6           5               1
7  115746    2           4               1
8  167791    3           2              5+
9  187645    2           4               1


,row_id,boardgame,strategygames,thematic,wargames,familygames,cgs,abstracts,partygames,childrensgames,rpgitem
0,224517,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,342942,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,161936,3,3,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,174430,4,5,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,397598,5,4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,316554,6,7,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,233078,7,6,4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,115746,8,<NA>,3,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,167791,9,8,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,187645,10,<NA>,5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [42]:
bgg_clean = bgg_data.copy()
print(bgg_clean.columns)

bgg_clean = bgg_clean.drop(columns=['min_playtime','max_playtime','description'])
bgg_clean = bgg_clean.merge(ranks_table[['row_id','boardgame']], on='row_id', how='left')
bgg_clean.rename(columns={'boardgame_y': 'rank_boardgame','boardgame_x': 'name'}, inplace=True)
bgg_clean.sort_values(by='rank_boardgame', inplace=True)
bgg_clean.tail(25)

bgg_expansions = bgg_clean[bgg_clean['type'] == 'boardgameexpansion'].copy()
bgg_games = bgg_clean[bgg_clean['type'] == 'boardgame'].copy()
bgg_games.shape
# bgg_expansions.shape

Index(['row_id', 'type', 'boardgame', 'description', 'min_players',
       'max_players', 'min_playtime', 'max_playtime', 'playingtime',
       'minimum_age', 'release_year', 'average_rating', 'num_of_ratings',
       'weight', 'num_of_weights', 'bayes_average', 'std_deviation',
       'language_dependency', 'owned', 'trading', 'wanting', 'wishing'],
      dtype='str')


(30452, 20)

In [43]:
# Export the main tables to CSV files
bgg_games.to_csv('data/source/bgg_games.csv', index=False)
bgg_expansions.to_csv('data/source/bgg_expansions.csv', index=False)
ranks_table.to_csv('data/source/ranks_table.csv', index=False)
suggested_numplayers_table.to_csv('data/source/suggested_numplayers_table.csv', index=False)
game_designers.to_csv('data/source/game_designers.csv', index=False)
game_artists.to_csv('data/source/game_artists.csv', index=False)
game_mechanics.to_csv('data/source/game_mechanics.csv', index=False)
game_categories.to_csv('data/source/game_categories.csv', index=False)
game_family.to_csv('data/source/game_family.csv', index=False)
game_publishers.to_csv('data/source/game_publishers.csv', index=False)

print("All tables exported successfully!")

All tables exported successfully!


In [44]:
bgg_games[bgg_games['playingtime']<5]
bgg_games[bgg_games['weight']<1]

,row_id,type,name,min_players,max_players,playingtime,minimum_age,release_year,average_rating,num_of_ratings,weight,num_of_weights,bayes_average,std_deviation,language_dependency,owned,trading,wanting,wishing,rank_boardgame
